## 1. Imports & Dataset Loading

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import root_mean_squared_error

from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

import lightgbm as lgb

df = pd.read_csv('../dataset/predict_prices_dataset.csv')
df.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,A1,2017,99,Manual,15735,Petrol,150,55.4,1.4,audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,audi


## 2. Baseline — All Features, Native Categorical Support, No Tuning

In [14]:
categorical_cols = ['Make', 'fuelType', 'transmission', 'model']

for col in categorical_cols:
    df[col] = df[col].astype('category')

x = df.drop(columns=['price'])
y = df['price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model = lgb.LGBMRegressor(
    objective='regression',
    learning_rate=0.01,
    n_estimators=2000,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(X = x_train, y = y_train )

y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")


               Train        Test
RMSE       1592.9961   1971.1787
MAE        1068.9751   1141.5192
R²            0.9705      0.9562


LightGBM handles categorical columns natively once their dtype is set to `category`, so no one-hot or target encoding is needed — the four categoricals (`Make`, `fuelType`, `transmission`, `model`) go straight into the model. Default-ish hyperparameters (`learning_rate=0.01`, `n_estimators=2000`, `num_leaves=31`), no tuning, no CV.

## Baseline Comparison

Before the categoricals were added, an earlier iteration of this baseline was run on **numeric features only** to measure the size of the categorical contribution:

| Setup | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:------|:--------:|:-------:|:---------:|:--------:|:------:|
| Numeric features only | 0.9136 | 0.8979 | 3008 | 1966 | 0.016 |
| **All features (this cell)** | **0.9705** | **0.9562** | **1971** | **1141** | **0.014** |

Adding the four categoricals cut Test RMSE by ~34% and lifted Test R² by ~6 points, without touching a single hyperparameter. Same pattern as the polynomial phase — categorical columns carry most of the price signal.

## 3. First Tuning — RandomizedSearchCV with a Wide Hyperparameter Space

In [ ]:
base_model = lgb.LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

param_distributions = {
    'n_estimators':      randint(200, 2000),
    'learning_rate':     uniform(0.005, 0.1),      
    'num_leaves':        randint(15, 128),
    'max_depth':         randint(3, 12),
    'min_child_samples': randint(5, 50),
    'subsample':         uniform(0.6, 0.4),       
    'colsample_bytree':  uniform(0.6, 0.4),       
    
    'reg_alpha':         uniform(0.0, 1.0),
    'reg_lambda':        uniform(0.0, 1.0),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=50,                  
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(x_train, y_train)


print("Cel mai bun scor CV (RMSE):", -search.best_score_)
print("Cei mai buni parametri:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

best_model = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Cel mai bun scor CV (RMSE): 1691.309749696391
Cei mai buni parametri:
  colsample_bytree: 0.7301598792637071
  learning_rate: 0.07964914051180243
  max_depth: 10
  min_child_samples: 11
  n_estimators: 1418
  num_leaves: 31
  reg_alpha: 0.3080607918523892
  reg_lambda: 0.5425402305548993
  subsample: 0.803525630735504


50 random combinations × 5-fold CV = 250 fits over a wide search space (`num_leaves` up to 128, `max_depth` up to 12, `learning_rate` up to 0.1, no lower bound on regularization). `RandomizedSearchCV` samples combinations from the distributions, evaluates each one with K-fold CV, and returns the best set. Neg-RMSE as scoring so the maximization direction points toward smaller error. The picked parameters lean expressive: `num_leaves=31`, `max_depth=10`, `min_child_samples=11`.

## 4. Evaluating the First Tuned Model

In [16]:
y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       1202.3023   1894.9162
MAE         833.9691   1049.7648
R²            0.9832      0.9595


Marginal gain on test (+0.003 R²), but the train↔test gap **widened**:

| Setup | Train R² | Test R² | Test RMSE | Gap R² |
|:------|:--------:|:-------:|:---------:|:------:|
| No tuning, all features | 0.9705 | 0.9562 | 1971 | 0.014 |
| **First tuning (wide space)** | 0.9832 | **0.9595** | **1895** | 0.024 |

The search optimizes the mean CV score, not the generalization gap — so it happily picks parameters that fit train harder if that raises CV a tiny bit. Classic sign that the tuning space allows too-expressive models.

## 5. Second Tuning — Constrained Space + Early Stopping

In [20]:
param_distributions = {
    'n_estimators':      randint(500, 3000),        
    'learning_rate':     uniform(0.005, 0.045),    
    'num_leaves':        randint(15, 40),
    'max_depth':         randint(3, 8),
    'min_child_samples': randint(20, 100),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
    'reg_alpha':         uniform(0.1, 1.9),        
    'reg_lambda':        uniform(0.1, 1.9),
}

x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

fit_params = {
    'eval_X': x_val,
    'eval_y': y_val,
    'eval_metric': 'rmse',
    'callbacks': [lgb.early_stopping(stopping_rounds=50, verbose=False)],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(x_tr, y_tr, **fit_params) 

print("Cel mai bun scor CV (RMSE):", -search.best_score_)
print("Cei mai buni parametri:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

best_model = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Cel mai bun scor CV (RMSE): 1825.4206703240877
Cei mai buni parametri:
  colsample_bytree: 0.6571467271687763
  learning_rate: 0.034289981282698376
  max_depth: 7
  min_child_samples: 21
  n_estimators: 2891
  num_leaves: 26
  reg_alpha: 1.8832501471299252
  reg_lambda: 0.10147965509792722
  subsample: 0.996884623716487


Two changes over the first tuning:
- **Restricted the search space toward simpler models** (`num_leaves` capped at 40, `max_depth` at 8, `min_child_samples` ≥ 20, regularization ≥ 0.1) — models likely to overfit shouldn't even be sampled.
- **Added early stopping** via `lgb.early_stopping(stopping_rounds=50)`. `n_estimators` becomes a *ceiling* — training halts if the eval metric doesn't improve for 50 rounds, so the actual tree count is data-driven instead of hyperparameter-driven.

The eval slice for early stopping is carved from the *training* set with a second `train_test_split(x_train, y_train, test_size=0.2)`, giving `x_tr` (used for the K-fold CV) and `x_val` (used only by the early-stopping callback). `x_test` stays completely untouched during tuning, so the final evaluation in the next cell is honest.

## 6. Evaluating the Second Tuned Model

In [21]:
y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       1419.1912   1733.0589
MAE         931.1727   1075.7016
R²            0.9766      0.9661


Best test score of the three tuning attempts, **and** the smallest gap:

| Setup | Train R² | Test R² | Test RMSE | Gap R² |
|:------|:--------:|:-------:|:---------:|:------:|
| No tuning | 0.9705 | 0.9562 | 1971 | 0.014 |
| First tuning (wide space) | 0.9832 | 0.9595 | 1895 | 0.024 |
| **Second tuning (constrained + ES)** | 0.9766 | **0.9661** | **1733** | **0.010** |

About +1 point of Test R² over the un-tuned baseline, gap cut in half. Hyperparameter tuning has real but modest headroom on the current feature set. The remaining lever is feature engineering.

## 7. Feature Engineering + Outlier Removal

In [ ]:
df['km_per_year'] = df['mileage'] / (2024 - df['year'] + 1)

df_clean = df[df['price'] >= 500]
df_clean = df_clean[df_clean['mpg'] <= 400]
df_clean = df_clean[df_clean['year'] >= 2001]


X = df_clean.drop(columns=['price'])
y = df_clean['price']

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = lgb.LGBMRegressor(
    **search.best_params_,
    objective='regression',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(x_train, y_train)

y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")


               Train        Test
RMSE       1362.0532   1708.2197
MAE         908.4249   1057.0919
R²            0.9784      0.9671


Two changes, both applied before the split:
- **`km_per_year = mileage / (2024 - year + 1)`** — a usage-intensity signal a tree can only build through many successive splits, given here directly.
- **Outlier trimming**: dropped rows with `price < 500`, `mpg > 400`, or `year < 2001` — same rules as the polynomial phase.

Reused the tuned hyperparameters from the previous step unchanged; no new tuning.

Result: another ~+1 pp on Test R² with the gap staying tight:

| Setup | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:------|:--------:|:-------:|:---------:|:--------:|:------:|
| Second tuning (no FE) | 0.9766 | 0.9661 | 1733 | 1076 | 0.010 |
| **+ km_per_year + outlier removal** | 0.9784 | **0.9671** | **1708** | **1057** | **0.011** |

## Full Progression

| Step | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:-----|:--------:|:-------:|:---------:|:--------:|:------:|
| Numeric only, no tuning | 0.914 | 0.898 | 3008 | 1966 | 0.016 |
| Numeric only, tuning A | 0.938 | 0.908 | 2854 | 1799 | 0.030 |
| Numeric only, tuning A+B | 0.917 | 0.901 | 2966 | 1932 | 0.016 |
| All features, no tuning | 0.9705 | 0.9562 | 1971 | 1141 | 0.014 |
| All features, tuning A | 0.9832 | 0.9595 | 1895 | 1050 | 0.024 |
| All features, tuning A+B | 0.9766 | 0.9661 | 1733 | 1076 | 0.010 |
| **+ km_per_year + outlier removal** | **0.9784** | **0.9671** | **1708** | **1057** | **0.011** |

The biggest jump is still the one seen in the polynomial phase: **adding the categoricals** (+6.5 pp of R², ~34% off Test RMSE). Everything after that stacks: tuning delivers +1 pp, and the feature-engineering step delivers another +1 pp on top — all without the train↔test gap opening up.

## Comparison with Polynomial Regression

The polynomial notebook ended at Test R² = 0.943 (degree 3 + Ridge alpha=1000, on the same features and same outlier rules). LightGBM:

| Model | Best Test R² | Best Test RMSE | Best Test MAE | Effort to reach it |
|:------|:------------:|:--------------:|:-------------:|:-------------------|
| Polynomial + Ridge (best) | 0.943 | 2253 | 1458 | Degree sweep, one-hot + target encoding, log/ratio experiments, alpha sweep |
| **LightGBM (default hyperparams, all features)** | **0.956** | **1971** | **1141** | Three lines: convert categoricals to `category`, fit, predict |
| **LightGBM (tuned, all features)** | **0.966** | **1733** | **1076** | + `RandomizedSearchCV` with early stopping on a val slice |
| **LightGBM (final, + FE + outlier removal)** | **0.967** | **1708** | **1057** | + `km_per_year` ratio, drop ~50 outlier rows |

Two things stand out:
1. **The polynomial's best score was beaten with default LightGBM hyperparameters and no feature engineering** — no one-hot, no target encoding, no polynomial expansion, no alpha sweep. Native categorical handling replaced all of it.
2. **The final LightGBM sits ~2.5 percentage points above the polynomial's ceiling** on Test R² and cut Test MAE from 1458 → 1057 (~28% lower), on the exact same features and outlier rules.

The polynomial notebook's closing line called it: *"Getting past ~0.95 would mean switching to a non-linear model like gradient boosting."* That's what happened here — the model family swap delivered more than any amount of polynomial-side feature engineering could have.